# HUẤN LUYỆN TINH CHỈNH TPS-INBETWEEN PHIÊN BẢN V1.1 (PRODUCTION RELEASE V1.1)
> **Mục tiêu Trọng tâm:** Đưa mô hình TPS-Inbetween đạt chuẩn sản phẩm hoàn thiện (Production Ready V1.1) trên tập dữ liệu chuẩn vàng quy mô lớn; triệt tiêu hoàn toàn rò rỉ RAM trên Kaggle Linux Container (CoW-Proof, No tmpfs Extract), mở khóa biên độ vẽ bù nét khuyết `detail_add`, tinh chỉnh đa tầng Contextnet/IFNet/UNet (Differential Learning Rates) và hiệu chỉnh hệ thống chấm điểm toàn diện với Dilated Stroke F1.  
> **Tập Dữ Liệu Nền Tảng:** `sakuga_golden_dataset_tps_ready.tar.gz` (4,165 Golden Triplets trích xuất từ 1,229 video Sakugabooru qua quy trình `sakuga_pipeline v2`).  
> **Kiến trúc Cốt lõi:** Adaptive Stroke Fusion Head + Gated Ghost Erasure Residual (ASF-GR Engine V1.1).  
> **Hàm Mất Mát:** 100% PyTorch GPU-Native Loss Engine (L1 + VGG + Stroke Dice + GPU Morphological Distance + Calibrated Ghost Penalty + Density).  
> **Nền tảng:** Kaggle Cloud GPU (NVIDIA Tesla T4 x2 Dual-GPU hoặc P100 16GB VRAM) - Tối ưu hóa song song đa GPU không nghẽn cổ chai CPU.

In [ ]:
# [CELL 1]: Cấu hình Môi trường, Phần cứng & Trình Dọn Rác Bộ Nhớ Chủ Động (V1.1)
import os
import sys
import ctypes

os.environ["MALLOC_TRIM_THRESHOLD_"] = "65536"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
# Giảm phân mảnh VRAM của CUDA caching allocator (phải đặt TRƯỚC khi import torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    if hasattr(sys.stdout, 'reconfigure'):
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    if hasattr(sys.stderr, 'reconfigure'):
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

import gc
import time
import math
import random
import json
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data.dataloader import default_collate
import torchvision.models as models

try:
    mp.set_sharing_strategy('file_descriptor')
except Exception:
    pass

def configure_glibc_allocator():
    try:
        libc = ctypes.CDLL('libc.so.6')
        # M_TRIM_THRESHOLD = -1 (64KB), M_ARENA_MAX = -8 (2)
        # [V1.1 FIX]: KHÔNG đặt M_MMAP_THRESHOLD (-3) nữa. Ép ngưỡng mmap 64KB khiến mỗi mảng ảnh nhỏ
        # bị cấp phát qua mmap riêng lẻ -> hàng vạn vùng nhớ phân mảnh, tăng RSS và chi phí page-fault.
        libc.mallopt(-1, 65536)
        libc.mallopt(-8, 2)
    except Exception:
        pass

configure_glibc_allocator()

# Thiết lập Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

# Kiểm tra GPU & Cấu hình thiết bị
NUM_GPUS = torch.cuda.device_count()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"[*] Đang sử dụng thiết bị: {device} | Số GPU khả dụng: {NUM_GPUS}")
for i in range(NUM_GPUS):
    prop = torch.cuda.get_device_properties(i)
    print(f"    - GPU {i}: {prop.name} (VRAM: {prop.total_memory / 1e9:.2f} GB)")

# Hàm dọn dẹp bộ nhớ chuyên sâu chống tràn RAM máy chủ và VRAM (Zero Memory Leak Engine)
def cleanup_memory():
    """
    Thu hồi rác Python, dọn bộ đệm CUDA VRAM và ép glibc trả lại RAM máy chủ cho Linux kernel.
    Giữ mức RAM của Kaggle luôn trong vùng an toàn (xanh lá cây), triệt tiêu báo động đỏ.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    try:
        # Ép glibc giải phóng triệt để các chunk nhớ heap trên Linux container về OS
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

# Giám sát RAM vật lý thực tế của container (Host RAM) để theo dõi hiện tượng RAM stack qua các epoch
def _read_cgroup_memory_bytes():
    """Đọc mức RAM container đang dùng từ cgroup v2 (memory.current) hoặc cgroup v1 (memory.usage_in_bytes)."""
    for p in ('/sys/fs/cgroup/memory.current', '/sys/fs/cgroup/memory/memory.usage_in_bytes'):
        try:
            with open(p, 'r') as f:
                return int(f.read().strip())
        except Exception:
            continue
    return None

def get_host_ram_gb():
    """
    Trả về (process_rss_gb, container_gb):
    - process_rss_gb: RSS của tiến trình chính + toàn bộ tiến trình con (DataLoader workers) qua psutil.
    - container_gb  : Mức tiêu thụ cgroup của toàn bộ container (giới hạn thực tế gây OOM trên Kaggle).
    Giá trị None nếu không đọc được.
    """
    rss_gb = None
    try:
        import psutil
        proc = psutil.Process(os.getpid())
        rss = proc.memory_info().rss
        for ch in proc.children(recursive=True):
            try:
                rss += ch.memory_info().rss
            except Exception:
                pass
        rss_gb = rss / (1024 ** 3)
    except Exception:
        pass
    cg = _read_cgroup_memory_bytes()
    cg_gb = cg / (1024 ** 3) if cg is not None else None
    return rss_gb, cg_gb

def format_host_ram():
    rss_gb, cg_gb = get_host_ram_gb()
    rss_s = f"{rss_gb:.2f} GB" if rss_gb is not None else "N/A"
    cg_s = f"{cg_gb:.2f} GB" if cg_gb is not None else "N/A"
    return f"Process RSS: {rss_s} | Container (cgroup): {cg_s}"

cleanup_memory()
print(f"[*] Host RAM hiện tại -> {format_host_ram()}")
print("[✓] CELL 1 hoàn tất: Môi trường và hàm quản lý bộ nhớ sâu sẵn sàng (V1.1).")

In [ ]:
# [CELL 2]: Tự Động Định Vị, Giải Nén Siêu Tốc & Thẩm Định Toàn Vẹn Golden Dataset V1.1
import os
import sys
import gc
import time
import subprocess
import tarfile
import pandas as pd

KAGGLE_INPUT = '/kaggle/input'
WORKING_DIR = '/kaggle/working'

# [PHƯƠNG ÁN TỐI ƯU V1.1 - FIX OOM]: TRÁNH OverlayFS ROOT (/kaggle/temp) VÀ TMPFS (/tmp)
# Trên Kaggle, '/' (bao gồm /kaggle/temp) là OverlayFS có upper-layer bị tính thẳng vào giới hạn RAM
# của container. Vì vậy thư mục giải nén dự phòng được đặt tại /kaggle/working/_temp_dataset
# (nằm trên ổ đĩa vật lý gắn ngoài). Thư mục này sẽ được XÓA ở CELL 12 trước khi kết thúc notebook
# để Kaggle không nén lại dữ liệu tạm khi Save Version.
TMP_DIR = os.path.join(WORKING_DIR, '_temp_dataset')
EXTRACT_DIR = os.path.join(TMP_DIR, 'sakuga_golden_dataset_tps_ready')
DATASET_EXTRACTED_TO_TEMP = False  # Cờ đánh dấu để CELL 12 biết cần dọn dẹp

DATASET_ROOT = None

# Bước 1: Quét đệ quy tìm trực tiếp thư mục chứa triplets.csv hoặc triplets_combined.csv
# Kaggle tự động giải nén file archive khi import dataset, dẫn tới cấu trúc lồng nhau nhiều cấp
# (Ví dụ: /kaggle/input/sakuga_golden_dataset_kaggle_upload/sakuga_golden_dataset_tps_ready/anime_tps_finetune_dataset_ready)
search_locations = [KAGGLE_INPUT, EXTRACT_DIR, TMP_DIR, WORKING_DIR, './data/processed', '../data/processed', '.']

print("[*] Đang quét tìm thư mục Golden Dataset (chứa triplets.csv)...")
for base_dir in search_locations:
    if not os.path.exists(base_dir):
        continue
    for root, dirs, files in os.walk(base_dir):
        if 'triplets.csv' in files or 'triplets_combined.csv' in files:
            DATASET_ROOT = root
            print(f"[✓] Tự động phát hiện thư mục Golden Dataset tại: {DATASET_ROOT}")
            break
    if DATASET_ROOT is not None:
        break

# Bước 2: Nếu chưa tìm thấy thư mục giải nén, quét tìm file archive (.tar.gz) và tự động giải nén vào NVMe
if DATASET_ROOT is None:
    print("[*] Chưa thấy thư mục giải nén, tiến hành quét tìm file nén (.tar.gz)...")
    tar_path = None
    for base_dir in search_locations:
        if not os.path.exists(base_dir):
            continue
        for root, dirs, files in os.walk(base_dir):
            for f in files:
                if f.endswith('.tar.gz') and ('sakuga' in f.lower() or 'golden' in f.lower() or 'tps' in f.lower()):
                    tar_path = os.path.join(root, f)
                    break
            if tar_path is not None:
                break
        if tar_path is not None:
            break

    if tar_path is not None and os.path.exists(tar_path):
        file_size_mb = os.path.getsize(tar_path) / (1024 * 1024)
        print(f"[✓] Phát hiện archive dataset: {tar_path} ({file_size_mb:.2f} MB)")
        print(f"[*] Tiến hành giải nén siêu tốc vào ổ đĩa vật lý /kaggle/working ({EXTRACT_DIR})...")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        DATASET_EXTRACTED_TO_TEMP = True
        t_start = time.time()
        
        extracted = False
        try:
            # Lệnh tar hệ thống trên Linux/Kaggle giải nén trực tiếp bằng C kernel (10 - 15 giây)
            cmd = ['tar', '-xzf', tar_path, '-C', EXTRACT_DIR]
            res = subprocess.run(cmd, capture_output=True, text=True)
            if res.returncode == 0:
                extracted = True
            else:
                print(f"[!] Lệnh tar báo lỗi: {res.stderr}. Chuyển sang fallback tarfile...")
        except Exception as e:
            print(f"[!] Không thể gọi subprocess tar: {e}. Chuyển sang fallback tarfile...")
            
        if not extracted:
            print("[*] Fallback sang thư viện tarfile chuẩn của Python...")
            with tarfile.open(tar_path, 'r:gz') as tar:
                tar.extractall(EXTRACT_DIR)
                
        extract_duration = time.time() - t_start
        print(f"[✓] Giải nén toàn bộ dataset hoàn tất chỉ trong {extract_duration:.1f}s!")
        
        # Quét lại EXTRACT_DIR để lấy chính xác thư mục chứa triplets.csv
        for root, dirs, files in os.walk(EXTRACT_DIR):
            if 'triplets.csv' in files or 'triplets_combined.csv' in files:
                DATASET_ROOT = root
                break

if DATASET_ROOT is None:
    raise FileNotFoundError("❌ Không tìm thấy tập dữ liệu Sakuga Golden Dataset! Vui lòng kiểm tra xem bạn đã thêm Dataset vào Kaggle Notebook chưa.")

# Nạp triplets metadata
triplets_csv_path = os.path.join(DATASET_ROOT, 'triplets.csv')
if not os.path.exists(triplets_csv_path):
    triplets_csv_path = os.path.join(DATASET_ROOT, 'triplets_combined.csv')

df_triplets = pd.read_csv(triplets_csv_path)
train_df = df_triplets[df_triplets['split'] == 'train'].reset_index(drop=True)
val_df = df_triplets[df_triplets['split'] == 'val'].reset_index(drop=True)

# Giữ lại số lượng mẫu dưới dạng int rồi giải phóng ngay DataFrame gốc (tránh worker fork kế thừa)
total_triplets_count = len(df_triplets)
num_train_samples = len(train_df)
num_val_samples = len(val_df)
del df_triplets
gc.collect()

print(f"[✓] Thư mục Golden Dataset hợp lệ: {DATASET_ROOT}")
print(f"[✓] Tổng số Golden Triplets: {total_triplets_count:,} mẫu")
print(f"    - Tập Huấn luyện (Train): {num_train_samples:,} mẫu ({num_train_samples/max(1, total_triplets_count)*100:.1f}%)")
print(f"    - Tập Thẩm định (Val)   : {num_val_samples:,} mẫu ({num_val_samples/max(1, total_triplets_count)*100:.1f}%)")
assert num_train_samples > 0 and num_val_samples > 0, "Dữ liệu train hoặc val bị rỗng! Kiểm tra triplets.csv."

# Kiểm tra thư mục matches
matches_dir = os.path.join(DATASET_ROOT, 'matches')
print(f"[*] Thư mục keypoint matches: {matches_dir} (Tồn tại: {os.path.exists(matches_dir)})")
print("[✓] CELL 2 hoàn tất: Toàn vẹn Golden Dataset được xác nhận (Tránh tmpfs /tmp & OverlayFS root).")

In [ ]:
# [CELL 3]: Hệ Thống Biến Dạng Lưới, 8 Điểm Neo Biên/Góc & Vector Hóa TPS Flow Batch
_NORM_GRID_CACHE = {}
_PIXEL_GRID_CACHE = {}
# [V1.1 FIX]: Giới hạn kích thước cache lưới (Bounded Eviction) để chống rò rỉ bộ nhớ khi gặp
# các ảnh Validation có độ phân giải khác nhau (mỗi (h, w, device, dtype) tạo 1 tensor lưới mới).
_MAX_GRID_CACHE_SIZE = 32

def warp_normalized_flow(img, flow):
    """
    Biến dạng lưới Coarse TPS (tọa độ chuẩn hóa [0, 1] sang [-1, 1]).
    Tối ưu hóa V1.0: Caching lưới cơ sở (1, H, W, 2) và cộng trực tiếp qua broadcasting
    (loại bỏ hoàn toàn meshgrid và .repeat(b, 1, 1, 1) dư thừa).
    """
    b, _, h, w = img.shape
    dev = flow.device
    dtype = flow.dtype
    key = (h, w, dev, dtype)
    # Giữ tham chiếu cục bộ (an toàn khi nhiều thread DataParallel cùng truy cập & clear cache)
    base_grid = _NORM_GRID_CACHE.get(key)
    if base_grid is None:
        if len(_NORM_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
            _NORM_GRID_CACHE.clear()
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, 1, h, device=dev, dtype=dtype),
            torch.linspace(0, 1, w, device=dev, dtype=dtype), indexing='ij'
        )
        base_grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0)
        _NORM_GRID_CACHE[key] = base_grid

    grid = (base_grid + flow.permute(0, 2, 3, 1)) * 2.0 - 1.0
    return F.grid_sample(img, grid, padding_mode='border', align_corners=True)

def warp_pixel_flow(img, flow):
    """
    Biến dạng lưới Fine Optical Flow IFNet (tọa độ pixel thực tế).
    Tối ưu hóa V1.0: Caching lưới cơ sở [-1, 1] và cộng trực tiếp qua broadcasting.
    """
    b, _, h, w = img.shape
    dev = flow.device
    dtype = flow.dtype
    key = (h, w, dev, dtype)
    base_grid = _PIXEL_GRID_CACHE.get(key)
    if base_grid is None:
        if len(_PIXEL_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
            _PIXEL_GRID_CACHE.clear()
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, h, device=dev, dtype=dtype),
            torch.linspace(-1, 1, w, device=dev, dtype=dtype), indexing='ij'
        )
        base_grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0)
        _PIXEL_GRID_CACHE[key] = base_grid

    flow_norm = torch.cat([
        flow[:, 0:1, :, :] / ((w - 1.0) / 2.0),
        flow[:, 1:2, :, :] / ((h - 1.0) / 2.0)
    ], dim=1)
    grid = base_grid + flow_norm.permute(0, 2, 3, 1)
    return F.grid_sample(img, grid, padding_mode='border', align_corners=True)

def tps_rbf(r):
    r = torch.clamp(r, min=1e-6)
    return (r ** 2) * torch.log(r)

# 8 Điểm neo tĩnh (4 góc + 4 điểm giữa biên) để khóa chặt khung hình, chống co rút viền ảnh
FIXED_ANCHORS_CPU = torch.tensor([
    [0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0],
    [0.5, 0.0], [0.5, 1.0], [0.0, 0.5], [1.0, 0.5]
], dtype=torch.float64)

FIXED_ANCHORS_NP = np.array([
    [0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0],
    [0.5, 0.0], [0.5, 1.0], [0.0, 0.5], [1.0, 0.5]
], dtype=np.float64)

def robust_tps_theta_from_points(c_src, c_dst, lambd=0.5, min_dist=0.015, add_anchors=True):
    """
    Giải hệ phương trình TPS đơn mẫu với 8 điểm neo biên/góc (kháng suy biến và chống co rút).
    """
    device = c_src.device
    n = c_src.shape[0]
    if n < 4:
        return torch.zeros((n + 3, 2), device=device)

    c_src_cpu = c_src.detach().cpu().to(torch.float64)
    c_dst_cpu = c_dst.detach().cpu().to(torch.float64)

    keep_indices = [0]
    for i in range(1, n):
        dists = torch.norm(c_src_cpu[i] - c_src_cpu[keep_indices], dim=-1)
        if torch.min(dists) >= min_dist:
            keep_indices.append(i)

    if len(keep_indices) < 4:
        return torch.zeros((n + 3, 2), device=device)

    c_s_filt = c_src_cpu[keep_indices]
    c_d_filt = c_dst_cpu[keep_indices]

    if add_anchors:
        c_s_filt = torch.cat([c_s_filt, FIXED_ANCHORS_CPU], dim=0)
        c_d_filt = torch.cat([c_d_filt, FIXED_ANCHORS_CPU], dim=0)

    n_filt = c_s_filt.shape[0]
    diff = c_s_filt.unsqueeze(1) - c_s_filt.unsqueeze(0)
    dist = torch.norm(diff, dim=-1)

    K = tps_rbf(dist) + lambd * torch.eye(n_filt, dtype=torch.float64)
    P = torch.cat([torch.ones((n_filt, 1), dtype=torch.float64), c_s_filt], dim=-1)
    L_top = torch.cat([K, P], dim=-1)
    L_bot = torch.cat([P.t(), torch.zeros((3, 3), dtype=torch.float64)], dim=-1)
    L = torch.cat([L_top, L_bot], dim=0)
    Y = torch.cat([c_d_filt, torch.zeros((3, 2), dtype=torch.float64)], dim=0)

    try:
        theta_filt = torch.linalg.solve(L, Y)
    except Exception:
        theta_filt = torch.linalg.lstsq(L, Y).solution

    return theta_filt.to(device=device, dtype=torch.float32), c_s_filt.to(device=device, dtype=torch.float32)

def solve_tps_single_cpu(c_s, c_d, max_kps=80, lambd=0.5, min_dist=0.015, max_pts=88, return_tensor=False):
    """
    Giải TPS đơn chiều hoàn toàn bằng NumPy.
    [V1.1 FIX]: Mặc định trả về (c_pad: np.ndarray float32, theta_pad: np.ndarray float32, valid: float)
    để DataLoader worker không phải quản lý đối tượng torch.Tensor (default_collate sẽ chuyển sang Tensor).
    Đặt return_tensor=True để nhận torch.Tensor (tương thích ngược khi gọi ngoài DataLoader).
    """
    if isinstance(c_s, torch.Tensor):
        c_s = c_s.detach().cpu().numpy().astype(np.float64, copy=False)
        c_d = c_d.detach().cpu().numpy().astype(np.float64, copy=False)
    else:
        c_s = np.asarray(c_s, dtype=np.float64)
        c_d = np.asarray(c_d, dtype=np.float64)

    def _out(c_pad, theta_pad, valid):
        if return_tensor:
            return torch.from_numpy(c_pad), torch.from_numpy(theta_pad), float(valid)
        return c_pad, theta_pad, float(valid)

    n = c_s.shape[0]
    if n < 4:
        return _out(np.zeros((max_pts, 2), dtype=np.float32), np.zeros((max_pts + 3, 2), dtype=np.float32), 0.0)

    # Lọc điểm quá gần trên mảng NumPy phẳng (không tạo PyTorch tensor trong vòng lặp)
    min_dist_sq = min_dist * min_dist
    keep_indices = [0]
    kept_pts = np.empty((n, 2), dtype=np.float64)
    kept_pts[0] = c_s[0]
    num_kept = 1

    for i in range(1, n):
        dx = kept_pts[:num_kept, 0] - c_s[i, 0]
        dy = kept_pts[:num_kept, 1] - c_s[i, 1]
        if np.min(dx * dx + dy * dy) >= min_dist_sq:
            kept_pts[num_kept] = c_s[i]
            keep_indices.append(i)
            num_kept += 1

    if num_kept < 4:
        return _out(np.zeros((max_pts, 2), dtype=np.float32), np.zeros((max_pts + 3, 2), dtype=np.float32), 0.0)

    if num_kept > max_kps:
        idx_sub = np.linspace(0, num_kept - 1, max_kps, dtype=int)
        keep_indices = [keep_indices[k] for k in idx_sub]

    c_s_filt = np.concatenate([c_s[keep_indices], FIXED_ANCHORS_NP], axis=0)
    c_d_filt = np.concatenate([c_d[keep_indices], FIXED_ANCHORS_NP], axis=0)

    m = c_s_filt.shape[0]
    diff = c_s_filt[:, None, :] - c_s_filt[None, :, :]
    dist = np.sqrt(np.sum(diff * diff, axis=-1))
    dist_clamped = np.maximum(dist, 1e-6)
    K = (dist_clamped ** 2) * np.log(dist_clamped) + lambd * np.eye(m, dtype=np.float64)

    P = np.concatenate([np.ones((m, 1), dtype=np.float64), c_s_filt], axis=-1)
    L_top = np.concatenate([K, P], axis=-1)
    L_bot = np.concatenate([P.T, np.zeros((3, 3), dtype=np.float64)], axis=-1)
    L = np.concatenate([L_top, L_bot], axis=0)
    Y = np.concatenate([c_d_filt, np.zeros((3, 2), dtype=np.float64)], axis=0)

    try:
        theta = np.linalg.solve(L, Y)
    except Exception:
        theta = np.linalg.lstsq(L, Y, rcond=None)[0]

    c_pad = np.zeros((max_pts, 2), dtype=np.float32)
    theta_pad = np.zeros((max_pts + 3, 2), dtype=np.float32)
    c_pad[:m] = c_s_filt.astype(np.float32)
    theta_pad[:m] = theta[:m].astype(np.float32)
    theta_pad[max_pts:max_pts+3] = theta[m:m+3].astype(np.float32)

    return _out(c_pad, theta_pad, 1.0)

def solve_tps_pair_cpu(kps, max_kps=80, lambd=0.5, min_dist=0.015, return_tensor=True):
    """
    Giải TPS 2 chiều (c0 -> c1 và c1 -> c0) trên CPU worker của DataLoader.
    Trả về c0_pad (88, 2), th01_pad (91, 2), c1_pad (88, 2), th10_pad (91, 2), valid_flag (scalar float32).
    - return_tensor=False: Toàn bộ là np.ndarray float32 (dùng bên trong DataLoader worker -> giảm overhead IPC).
    - return_tensor=True : torch.Tensor (mặc định, tương thích ngược cho nhánh fallback inference của mô hình).
    """
    MAX_PTS = 88
    if kps is None or len(kps) < 2 or (hasattr(kps, 'shape') and len(kps.shape) > 1 and kps.shape[1] < 4):
        z_c0 = np.zeros((MAX_PTS, 2), dtype=np.float32)
        z_th01 = np.zeros((MAX_PTS + 3, 2), dtype=np.float32)
        z_c1 = np.zeros((MAX_PTS, 2), dtype=np.float32)
        z_th10 = np.zeros((MAX_PTS + 3, 2), dtype=np.float32)
        z_v = np.array(0.0, dtype=np.float32)
        if return_tensor:
            return (torch.from_numpy(z_c0), torch.from_numpy(z_th01), torch.from_numpy(z_c1),
                    torch.from_numpy(z_th10), torch.from_numpy(z_v))
        return z_c0, z_th01, z_c1, z_th10, z_v

    # Truyền trực tiếp mảng kps[0] và kps[1] vào solve_tps_single_cpu mà không cần tạo tensor trung gian
    c0 = kps[0]
    c1 = kps[1]

    c0_pad, th01_pad, v01 = solve_tps_single_cpu(c0, c1, max_kps=max_kps, lambd=lambd, min_dist=min_dist, max_pts=MAX_PTS, return_tensor=False)
    c1_pad, th10_pad, v10 = solve_tps_single_cpu(c1, c0, max_kps=max_kps, lambd=lambd, min_dist=min_dist, max_pts=MAX_PTS, return_tensor=False)

    v = np.array(1.0 if (v01 > 0.5 and v10 > 0.5) else 0.0, dtype=np.float32)
    if return_tensor:
        return (torch.from_numpy(c0_pad), torch.from_numpy(th01_pad), torch.from_numpy(c1_pad),
                torch.from_numpy(th10_pad), torch.from_numpy(v))
    return c0_pad, th01_pad, c1_pad, th10_pad, v

_GRID_CACHE = {}

def get_grid_pts(h, w, device):
    key = (h, w, device)
    pts = _GRID_CACHE.get(key)
    if pts is None:
        if len(_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
            _GRID_CACHE.clear()
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, 1, h, device=device),
            torch.linspace(0, 1, w, device=device), indexing='ij'
        )
        pts = torch.stack([grid_x, grid_y], dim=-1).view(-1, 2)
        _GRID_CACHE[key] = pts
    return pts

@torch.no_grad()
def tps_flow_from_theta_batch(theta_batch, c_src_batch, valid_mask, h, w, device, chunk_size=65536):
    """
    Vector hóa 100% trên GPU: Tính toán trường dòng chuyển động TPS cho TOÀN BỘ BATCH cùng lúc
    bằng torch.cdist và torch.bmm (Bảo toàn 100% giá trị toán học, nhanh gấp 4 lần, không tạo tensor 4D).
    - theta_batch: (B, 91, 2)
    - c_src_batch: (B, 88, 2)
    - valid_mask: (B, 1, 1, 1)
    - Output: flow (B, 2, H, W)
    """
    with torch.amp.autocast('cuda', enabled=False):
        theta_f32 = theta_batch.float()
        c_src_f32 = c_src_batch.float()
        valid_f32 = valid_mask.float()

        b = theta_f32.shape[0]
        pts = get_grid_pts(h, w, device)
        total_pts = pts.shape[0]
        flow_chunks = []

        for i in range(0, total_pts, chunk_size):
            p_chunk = pts[i:i + chunk_size]
            c_len = p_chunk.shape[0]
            pts_exp = p_chunk.unsqueeze(0).expand(b, -1, -1)

            # Dùng torch.cdist thay vì tạo tensor 4D (B, C, 88, 2) tốn VRAM
            dist = torch.cdist(pts_exp, c_src_f32, p=2.0)
            U = tps_rbf(dist)

            ones = torch.ones((b, c_len, 1), device=device, dtype=pts.dtype)
            P = torch.cat([ones, pts_exp], dim=-1)
            basis = torch.cat([U, P], dim=-1)

            mapped = torch.bmm(basis, theta_f32)
            flow_chunks.append(mapped - pts_exp)

        flow = torch.cat(flow_chunks, dim=1)
        flow = flow.view(b, h, w, 2).permute(0, 3, 1, 2).contiguous()
        flow.mul_(valid_f32.view(b, 1, 1, 1))
        return flow

def tps_flow_from_theta(theta, c_src, h, w, device, chunk_size=32768):
    """Hàm tính TPS đơn lẻ (Fallback tương thích ngược cho inference)."""
    n = c_src.shape[0]
    if n < 4 or theta.shape[0] != (n + 3):
        return torch.zeros((1, 2, h, w), device=device)

    pts = get_grid_pts(h, w, device)
    total_pts = pts.shape[0]
    flow_chunks = []

    for i in range(0, total_pts, chunk_size):
        p_chunk = pts[i:i + chunk_size]
        diff = p_chunk.unsqueeze(1) - c_src.unsqueeze(0)
        dist = torch.norm(diff, dim=-1)
        U = tps_rbf(dist)
        P = torch.cat([torch.ones((p_chunk.shape[0], 1), device=device), p_chunk], dim=-1)
        basis = torch.cat([U, P], dim=-1)
        mapped = torch.matmul(basis, theta)
        flow_chunk = mapped - p_chunk
        flow_chunks.append(flow_chunk)

    flow = torch.cat(flow_chunks, dim=0)
    flow = flow.view(h, w, 2).permute(2, 0, 1).unsqueeze(0)
    return flow

print("[✓] CELL 3 hoàn tất: Warp Layers Cache, 8 Anchors & Vectorized TPS Batch Solver sẵn sàng.")

In [ ]:
# [CELL 4]: Dataset & DataLoader An Toàn Bộ Nhớ Chia Sẻ & Giải TPS Nền Cho Golden Dataset V1.1
def _worker_init_fn(worker_id):
    configure_glibc_allocator()
    torch.set_num_threads(1)
    cv2.setNumThreads(0)
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

class SakugaGoldenDataset_v1(Dataset):
    def __init__(self, root_dir, df_split, is_train=True, crop_size=512):
        self.root_dir = root_dir
        self.is_train = is_train
        self.crop_size = crop_size
        self.split_name = 'train' if is_train else 'val'
        # [V1.1 FIX]: Đã XÓA self._access_count. Ghi vào thuộc tính đối tượng + gọi gc.collect() bên trong
        # worker (fork) làm bẩn các trang nhớ chứa PyGC_Head -> kích hoạt Copy-on-Write hàng loạt.
        self.length = len(df_split)

        # 1. Mã hóa toàn bộ đường dẫn ảnh và match_file thành mảng NumPy bytes cố định (dtype='S256')
        # Loại bỏ hoàn toàn mảng Python list / chuỗi Unicode động để triệt tiêu Copy-on-Write (CoW)
        p0_bytes = []
        pgt_bytes = []
        p1_bytes = []
        pmatch_bytes = []

        for _, row in df_split.iterrows():
            f0 = str(row['frame0']).lstrip('/\\')
            fgt = str(row['frame_gt']).lstrip('/\\')
            f1 = str(row['frame1']).lstrip('/\\')
            fm = str(row['match_file']).lstrip('/\\')

            path0 = os.path.join(root_dir, f0)
            if not os.path.exists(path0):
                path0 = os.path.join(root_dir, self.split_name, f0)
                pathgt = os.path.join(root_dir, self.split_name, fgt)
                path1 = os.path.join(root_dir, self.split_name, f1)
                pathm = os.path.join(root_dir, 'matches', fm)
            else:
                pathgt = os.path.join(root_dir, fgt)
                path1 = os.path.join(root_dir, f1)
                pathm = os.path.join(root_dir, fm)

            p0_bytes.append(path0.encode('utf-8'))
            pgt_bytes.append(pathgt.encode('utf-8'))
            p1_bytes.append(path1.encode('utf-8'))
            pmatch_bytes.append(pathm.encode('utf-8'))

        self.p0_arr = np.array(p0_bytes, dtype='S256')
        self.pgt_arr = np.array(pgt_bytes, dtype='S256')
        self.p1_arr = np.array(p1_bytes, dtype='S256')
        self.pmatch_arr = np.array(pmatch_bytes, dtype='S256')
        self.t_arr = df_split['t'].to_numpy(dtype=np.float32, copy=True)
        self.clips_arr = np.array([str(c).encode('utf-8') for c in df_split['clip_id']], dtype='S64')

        # 2. [TỐI ƯU HÓA CO-W]: Cấp phát 1 mảng NumPy 4D liền mạch duy nhất (length, 2, MAX_KPS_CACHE, 2)
        # và 1 mảng 1D kps_count để thay thế hoàn toàn Python list kps_cache
        MAX_KPS_CACHE = 256
        self.kps_data = np.zeros((self.length, 2, MAX_KPS_CACHE, 2), dtype=np.float32)
        self.kps_count = np.zeros(self.length, dtype=np.int32)

        print(f"[*] Preloading keypoints cache vào mảng NumPy 4D liền mạch ({self.split_name}: {self.length} mẫu)...")
        for idx in range(self.length):
            pmatch = self.pmatch_arr[idx].decode('utf-8')
            if os.path.exists(pmatch):
                try:
                    kps = np.load(pmatch).astype(np.float32)
                    if kps.ndim == 3 and kps.shape[0] == 2 and kps.shape[2] == 2:
                        n = kps.shape[1]
                        if n > MAX_KPS_CACHE:
                            idx_sub = np.linspace(0, n - 1, MAX_KPS_CACHE, dtype=int)
                            self.kps_data[idx] = kps[:, idx_sub, :]
                            self.kps_count[idx] = MAX_KPS_CACHE
                        elif n > 0:
                            self.kps_data[idx, :, :n, :] = kps
                            self.kps_count[idx] = n
                except Exception:
                    pass
        print(f"[✓] Đã nạp xong keypoints cache vào mảng NumPy liền mạch (CoW-Proof).")

        # 3. [TỐI ƯU HÓA CO-W]: Lưu TPS Cache Validation thành 5 mảng NumPy phẳng contiguous
        self.val_c0 = np.zeros((self.length, 88, 2), dtype=np.float32)
        self.val_th01 = np.zeros((self.length, 91, 2), dtype=np.float32)
        self.val_c1 = np.zeros((self.length, 88, 2), dtype=np.float32)
        self.val_th10 = np.zeros((self.length, 91, 2), dtype=np.float32)
        self.val_valid = np.zeros((self.length, 1, 1, 1), dtype=np.float32)

        if not self.is_train:
            print(f"[*] Precomputing TPS cache vào 5 mảng NumPy cho tập Validation ({self.length} mẫu)...")
            for idx in range(self.length):
                p0 = self.p0_arr[idx].decode('utf-8')
                cnt = int(self.kps_count[idx])
                kps = self.kps_data[idx, :, :cnt, :].copy() if cnt > 0 else np.zeros((2, 0, 2), dtype=np.float32)
                try:
                    with Image.open(p0) as img_hdr:
                        w, h = img_hdr.size
                except Exception:
                    w, h = 512, 512
                pad_h = ((h + 31) // 32) * 32
                pad_w = ((w + 31) // 32) * 32
                if kps is not None and kps.shape[1] > 0:
                    kps_norm = kps.copy()
                    kps_norm[0, :, 0] = (kps_norm[0, :, 0] * w) / pad_w
                    kps_norm[0, :, 1] = (kps_norm[0, :, 1] * h) / pad_h
                    kps_norm[1, :, 0] = (kps_norm[1, :, 0] * w) / pad_w
                    kps_norm[1, :, 1] = (kps_norm[1, :, 1] * h) / pad_h
                else:
                    kps_norm = kps
                c0_p, th01_p, c1_p, th10_p, v = solve_tps_pair_cpu(kps_norm, max_kps=80, lambd=0.5, min_dist=0.015, return_tensor=False)
                self.val_c0[idx] = c0_p
                self.val_th01[idx] = th01_p
                self.val_c1[idx] = c1_p
                self.val_th10[idx] = th10_p
                self.val_valid[idx, 0, 0, 0] = float(v)
            print(f"[✓] Đã precompute xong {self.length} mẫu TPS Validation vào contiguous NumPy buffers.")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # [V1.1 FIX]: KHÔNG gọi gc.collect()/malloc_trim bên trong worker, KHÔNG ghi thuộc tính self.*
        p0 = self.p0_arr[idx].decode('utf-8')
        pgt = self.pgt_arr[idx].decode('utf-8')
        p1 = self.p1_arr[idx].decode('utf-8')
        # Pha Train không dùng clip_name -> trả về chỉ số int (giảm cấp phát chuỗi Unicode qua IPC queue)
        clip_name = int(idx) if self.is_train else self.clips_arr[idx].decode('utf-8')

        im0 = cv2.imread(p0, cv2.IMREAD_GRAYSCALE)
        imgt = cv2.imread(pgt, cv2.IMREAD_GRAYSCALE)
        im1 = cv2.imread(p1, cv2.IMREAD_GRAYSCALE)

        if im0 is None or imgt is None or im1 is None:
            cs = self.crop_size if self.is_train else 512
            im0 = np.full((cs, cs), 255, dtype=np.uint8)
            imgt = np.full((cs, cs), 255, dtype=np.uint8)
            im1 = np.full((cs, cs), 255, dtype=np.uint8)

        h, w = im0.shape

        if self.is_train:
            cnt = int(self.kps_count[idx])
            kps = self.kps_data[idx, :, :cnt, :].copy() if cnt > 0 else np.zeros((2, 0, 2), dtype=np.float32)
            cs = self.crop_size
            if h > cs and w > cs:
                top = random.randint(0, h - cs)
                left = random.randint(0, w - cs)
                im0_crop = im0[top:top+cs, left:left+cs]
                imggt_crop = imgt[top:top+cs, left:left+cs]
                im1_crop = im1[top:top+cs, left:left+cs]

                if kps.shape[1] > 0:
                    kps[0, :, 0] = (kps[0, :, 0] * w - left) / cs
                    kps[0, :, 1] = (kps[0, :, 1] * h - top) / cs
                    kps[1, :, 0] = (kps[1, :, 0] * w - left) / cs
                    kps[1, :, 1] = (kps[1, :, 1] * h - top) / cs
                    mask0 = (kps[0, :, 0] >= 0) & (kps[0, :, 0] <= 1) & (kps[0, :, 1] >= 0) & (kps[0, :, 1] <= 1)
                    mask1 = (kps[1, :, 0] >= 0) & (kps[1, :, 0] <= 1) & (kps[1, :, 1] >= 0) & (kps[1, :, 1] <= 1)
                    kps = kps[:, mask0 & mask1, :]
            else:
                pad_h = ((h + 31) // 32) * 32
                pad_w = ((w + 31) // 32) * 32
                im0_crop = np.pad(im0, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
                imggt_crop = np.pad(imgt, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
                im1_crop = np.pad(im1, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)

            img0 = (255 - im0_crop).astype(np.float32) * (1.0 / 255.0)
            imggt = (255 - imggt_crop).astype(np.float32) * (1.0 / 255.0)
            img1 = (255 - im1_crop).astype(np.float32) * (1.0 / 255.0)

            c0_pad, th01_pad, c1_pad, th10_pad, valid_flag = solve_tps_pair_cpu(kps, max_kps=80, lambd=0.5, min_dist=0.015, return_tensor=False)
            valid_flag = np.full((1, 1, 1), float(valid_flag), dtype=np.float32)
        else:
            pad_h = ((h + 31) // 32) * 32
            pad_w = ((w + 31) // 32) * 32
            im0_pad = np.pad(im0, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
            imggt_pad = np.pad(imgt, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
            im1_pad = np.pad(im1, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)

            img0 = (255 - im0_pad).astype(np.float32) * (1.0 / 255.0)
            imggt = (255 - imggt_pad).astype(np.float32) * (1.0 / 255.0)
            img1 = (255 - im1_pad).astype(np.float32) * (1.0 / 255.0)

            # Lấy trực tiếp từ contiguous NumPy arrays bằng .copy(), 0% CoW (trả về NumPy thuần)
            c0_pad = self.val_c0[idx].copy()
            th01_pad = self.val_th01[idx].copy()
            c1_pad = self.val_c1[idx].copy()
            th10_pad = self.val_th10[idx].copy()
            valid_flag = self.val_valid[idx].copy()

        # [V1.1 FIX]: Trả về mảng NumPy contiguous float32; default_collate sẽ chuyển sang Tensor ở bước cuối
        t0 = np.ascontiguousarray(img0[None, :, :], dtype=np.float32)
        tgt = np.ascontiguousarray(imggt[None, :, :], dtype=np.float32)
        t1 = np.ascontiguousarray(img1[None, :, :], dtype=np.float32)
        t_val = np.full((1, 1, 1), float(self.t_arr[idx]), dtype=np.float32)
        c0_pad = np.ascontiguousarray(c0_pad, dtype=np.float32)
        th01_pad = np.ascontiguousarray(th01_pad, dtype=np.float32)
        c1_pad = np.ascontiguousarray(c1_pad, dtype=np.float32)
        th10_pad = np.ascontiguousarray(th10_pad, dtype=np.float32)
        valid_flag = np.ascontiguousarray(valid_flag, dtype=np.float32)

        return t0, tgt, t1, t_val, c0_pad, th01_pad, c1_pad, th10_pad, valid_flag, clip_name

custom_collate_fn_v1 = default_collate

PER_GPU_BATCH = 8
BATCH_SIZE = PER_GPU_BATCH * max(1, NUM_GPUS)
VAL_BATCH_SIZE = max(1, NUM_GPUS)

train_dataset = SakugaGoldenDataset_v1(DATASET_ROOT, train_df, is_train=True, crop_size=512)
val_dataset = SakugaGoldenDataset_v1(DATASET_ROOT, val_df, is_train=False)

# [V1.1 FIX]: Xóa Pandas DataFrame TRƯỚC KHI tạo DataLoader để worker fork không kế thừa chúng
del train_df, val_df
cleanup_memory()

# [V1.1 FIX]: persistent_workers=False -> Sau mỗi epoch 2 worker bị hủy, Linux thu hồi 100% heap,
# bộ đệm OpenCV/LAPACK và các trang Copy-on-Write của chúng (tạo lại worker chỉ mất ~0.2s/epoch).
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    persistent_workers=False,
    pin_memory=False,
    prefetch_factor=2,
    collate_fn=custom_collate_fn_v1,
    worker_init_fn=_worker_init_fn,
    drop_last=True
)

val_fast_subset = Subset(val_dataset, range(min(len(val_dataset), 100 * VAL_BATCH_SIZE)))

# Validation loaders: num_workers=0, persistent_workers=False để giải phóng 4 tiến trình ngầm & GB RAM IPC
val_loader_fast = DataLoader(
    val_fast_subset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    collate_fn=custom_collate_fn_v1
)

val_loader_full = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    collate_fn=custom_collate_fn_v1
)

val_loader = val_loader_full

print(f"[✓] DataLoaders Golden Dataset V1.1 sẵn sàng: {len(train_loader)} batches (Train), {len(val_loader_fast)} batches (Val Fast), {len(val_loader_full)} batches (Val Full).")
print("[✓] CELL 4 hoàn tất: Dataset & DataLoaders triệt tiêu 100% Copy-on-Write (CoW).")
print(f"[*] Host RAM hiện tại -> {format_host_ram()}")

In [ ]:
# [CELL 5]: Kiến Trúc TPS-Inbetween V1.1 (ASF-GR Engine - Không Nghẽn Cổ Chai GPU)
def conv(in_p, out_p, kernel_size=3, stride=1, padding=1, dilation=1):
    return nn.Sequential(
        nn.Conv2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=True),
        nn.PReLU(out_p)
    )

def deconv(in_p, out_p, kernel_size=4, stride=2, padding=1):
    return nn.Sequential(
        nn.ConvTranspose2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, bias=True),
        nn.PReLU(out_p)
    )

class Conv2(nn.Module):
    def __init__(self, in_p, out_p, stride=2):
        super(Conv2, self).__init__()
        self.conv1 = conv(in_p, out_p, 3, stride, 1)
        self.conv2 = conv(out_p, out_p, 3, 1, 1)
    def forward(self, x):
        return self.conv2(self.conv1(x))

class Contextnet(nn.Module):
    def __init__(self, indim=1, c=16):
        super(Contextnet, self).__init__()
        self.conv1 = Conv2(indim, c)
        self.conv2 = Conv2(c, 2*c)
        self.conv3 = Conv2(2*c, 4*c)
        self.conv4 = Conv2(4*c, 8*c)
    def forward(self, x, flow):
        x = self.conv1(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f1 = warp_normalized_flow(x, flow)
        x = self.conv2(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f2 = warp_normalized_flow(x, flow)
        x = self.conv3(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f3 = warp_normalized_flow(x, flow)
        x = self.conv4(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f4 = warp_normalized_flow(x, flow)
        return [f1, f2, f3, f4]

class IFBlock(nn.Module):
    def __init__(self, in_planes, c=64):
        super(IFBlock, self).__init__()
        self.conv0 = nn.Sequential(
            conv(in_planes, c//2, 3, 2, 1),
            conv(c//2, c, 3, 2, 1),
        )
        self.convblock = nn.Sequential(*[conv(c, c) for _ in range(8)])
        self.lastconv = nn.ConvTranspose2d(c, 5, 4, 2, 1)

    def forward(self, x, flow, scale):
        if scale != 1:
            x = F.interpolate(x, scale_factor=1./scale, mode='bilinear', align_corners=False)
        if flow is not None:
            flow = F.interpolate(flow, scale_factor=1./scale, mode='bilinear', align_corners=False) * (1./scale)
            x = torch.cat((x, flow), 1)
        x = self.conv0(x)
        x = self.convblock(x) + x
        tmp = self.lastconv(x)
        tmp = F.interpolate(tmp, scale_factor=scale * 2, mode='bilinear', align_corners=False)
        flow = tmp[:, :4] * scale * 2
        mask = tmp[:, 4:5]
        return flow, mask

class IFNet(nn.Module):
    def __init__(self):
        super(IFNet, self).__init__()
        self.block0 = IFBlock(6, c=240)
        self.block1 = IFBlock(13+4, c=150)
        self.block2 = IFBlock(13+4, c=90)

    def forward(self, I0t, I1t, scale=[4, 2, 1]):
        x = torch.cat((I0t.repeat(1, 3, 1, 1), I1t.repeat(1, 3, 1, 1)), dim=1)
        img0 = x[:, :3]
        img1 = x[:, 3:6]
        flow_list = []
        mask_list = []
        warped_img0 = img0
        warped_img1 = img1
        flow = None
        stu = [self.block0, self.block1, self.block2]
        for i in range(3):
            if flow is not None:
                flow_d, mask_d = stu[i](torch.cat((img0, img1, warped_img0, warped_img1, mask), 1), flow, scale=scale[i])
                flow = flow + flow_d
                mask = mask + mask_d
            else:
                flow, mask = stu[i](torch.cat((img0, img1), 1), None, scale=scale[i])
            mask_list.append(torch.sigmoid(mask))
            flow_list.append(flow)
            warped_img0 = warp_pixel_flow(img0, flow[:, :2])
            warped_img1 = warp_pixel_flow(img1, flow[:, 2:4])
        return flow_list[2], mask_list[2]

# [ĐỘT PHÁ 1]: Adaptive Stroke Fusion Head (ASF Head) - Triệt tiêu Hard Blending
class AdaptiveStrokeFusionHead(nn.Module):
    def __init__(self):
        super(AdaptiveStrokeFusionHead, self).__init__()
        self.net = nn.Sequential(
            conv(8, 32, 3, 1, 1),
            conv(32, 32, 3, 1, 1),
            nn.Conv2d(32, 1, 3, 1, 1)
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, fine_I0t, fine_I1t, flow, ifnet_mask, t_vals):
        b, _, h, w = fine_I0t.shape
        t_map = t_vals.repeat(1, 1, h, w)
        inp = torch.cat([fine_I0t, fine_I1t, flow, ifnet_mask, t_map], dim=1)
        residual_logit = self.net(inp)
        prior_logit = torch.logit(torch.clamp(1.0 - t_map, 1e-4, 1.0 - 1e-4))
        blend_mask = torch.sigmoid(prior_logit + residual_logit)
        return blend_mask

# [ĐỘT PHÁ 2]: Synthesis UNet v1.1 với Gated Ghost Erasure (GGER) & Unlocked Detail Add
class SynthesisUnet_v1(nn.Module):
    def __init__(self, indim=9, c=16):
        super(SynthesisUnet_v1, self).__init__()
        self.down0 = Conv2(indim, 2*c)
        self.down1 = Conv2(4*c, 4*c)
        self.down2 = Conv2(8*c, 8*c)
        self.down3 = Conv2(16*c, 16*c)
        self.up0 = deconv(32*c, 8*c)
        self.up1 = deconv(16*c, 4*c)
        self.up2 = deconv(8*c, 2*c)
        self.up3 = deconv(4*c, c)

        self.conv_erase = nn.Conv2d(c, 1, 3, 1, 1)
        nn.init.zeros_(self.conv_erase.weight)
        nn.init.constant_(self.conv_erase.bias, 2.0)  # sigmoid(2.0) ~ 0.88

        self.conv_add = nn.Conv2d(c, 1, 3, 1, 1)
        nn.init.zeros_(self.conv_add.weight)
        nn.init.constant_(self.conv_add.bias, -3.0)

    def forward(self, img0, img1, flow0, flow1, warped0, warped1, mask, c0, c1):
        s0 = self.down0(torch.cat((img0, img1, warped0, warped1, flow0, flow1, mask), 1))
        s1 = self.down1(torch.cat((s0, c0[0], c1[0]), 1))
        s2 = self.down2(torch.cat((s1, c0[1], c1[1]), 1))
        s3 = self.down3(torch.cat((s2, c0[2], c1[2]), 1))
        x = self.up0(torch.cat((s3, c0[3], c1[3]), 1))
        x = self.up1(torch.cat((x, s2), 1))
        x = self.up2(torch.cat((x, s1), 1))
        x = self.up3(torch.cat((x, s0), 1))

        gate_erase = torch.sigmoid(self.conv_erase(x))
        detail_add = torch.sigmoid(self.conv_add(x))
        return gate_erase, detail_add

class TPSInbetweenFinetune_v1(nn.Module):
    """
    Kiến trúc TPS-Inbetween V1.0 hoàn chỉnh.
    Bảo toàn 100% cấu trúc state_dict (encoder, IFNet, asf_head, unet) để tương thích tuyệt đối.
    """
    def __init__(self):
        super(TPSInbetweenFinetune_v1, self).__init__()
        self.encoder = Contextnet(indim=1)
        self.IFNet = IFNet()
        self.asf_head = AdaptiveStrokeFusionHead()
        self.unet = SynthesisUnet_v1(indim=9)

    def forward(self, x0, x1, t_vals, kps_input=None, kps_counts=None, max_kps=80, tps_precomputed=None):
        b, c, h, w = x0.shape
        dev = x0.device
        t_vals = t_vals.view(b, 1, 1, 1).to(dev)

        # 1. Coarse TPS Motion
        # [TỐI ƯU HÓA ĐỘT PHÁ V1.0]: Nếu đã có tps_precomputed từ DataLoader -> Chạy 100% GPU BMM Batch
        if tps_precomputed is not None:
            c0_b, th01_b, c1_b, th10_b, valid_b = tps_precomputed
            c0_b = c0_b.to(dev)
            th01_b = th01_b.to(dev)
            c1_b = c1_b.to(dev)
            th10_b = th10_b.to(dev)
            valid_b = valid_b.to(dev)

            coarse_flow01 = tps_flow_from_theta_batch(th01_b, c0_b, valid_b, h, w, dev)
            coarse_flow10 = tps_flow_from_theta_batch(th10_b, c1_b, valid_b, h, w, dev)
        else:
            # Nhánh Fallback tương thích ngược cho inference bên ngoài
            flow01_list, flow10_list = [], []
            with torch.no_grad():
                for i in range(b):
                    kps = kps_input[i] if kps_input is not None else None
                    c0_p, th01_p, c1_p, th10_p, v = solve_tps_pair_cpu(kps, max_kps=max_kps, lambd=0.5)
                    c0_p = c0_p.unsqueeze(0).to(dev)
                    th01_p = th01_p.unsqueeze(0).to(dev)
                    c1_p = c1_p.unsqueeze(0).to(dev)
                    th10_p = th10_p.unsqueeze(0).to(dev)
                    v_t = v.view(1, 1, 1, 1).to(dev)

                    f01 = tps_flow_from_theta_batch(th01_p, c0_p, v_t, h, w, dev)
                    f10 = tps_flow_from_theta_batch(th10_p, c1_p, v_t, h, w, dev)
                    flow01_list.append(f01)
                    flow10_list.append(f10)

            coarse_flow01 = torch.cat(flow01_list, dim=0)
            coarse_flow10 = torch.cat(flow10_list, dim=0)

        coarse_flow0t = coarse_flow01 * (t_vals ** 2) - (1.0 - t_vals) * t_vals * coarse_flow10
        coarse_flow1t = coarse_flow10 * ((1.0 - t_vals) ** 2) - t_vals * (1.0 - t_vals) * coarse_flow01

        # 2. Coarse Warping
        coarse_I0t = warp_normalized_flow(x0, coarse_flow0t)
        coarse_I1t = warp_normalized_flow(x1, coarse_flow1t)

        # 3. Fine Motion Refinement (Cho phép gradient truyền qua Contextnet khi fine-tune V1.1)
        feats0 = self.encoder(x0, coarse_flow0t)
        feats1 = self.encoder(x1, coarse_flow1t)
        flow_refine, ifnet_mask = self.IFNet(coarse_I0t, coarse_I1t)

        # 4. Fine Warping
        fine_I0t = warp_pixel_flow(coarse_I0t, flow_refine[:, :2])
        fine_I1t = warp_pixel_flow(coarse_I1t, flow_refine[:, 2:4])

        # 5. Adaptive Stroke Fusion (ASF Head)
        blend_mask = self.asf_head(fine_I0t, fine_I1t, flow_refine, ifnet_mask, t_vals)
        base_interp = blend_mask * fine_I0t + (1.0 - blend_mask) * fine_I1t

        # 6. Gated Ghost Erasure Residual (Synthesis UNet v1.0)
        gate_erase, detail_add = self.unet(
            x0, x1,
            coarse_flow0t, coarse_flow1t,
            fine_I0t, fine_I1t, ifnet_mask, feats0, feats1
        )
        pred_t = torch.clamp(gate_erase * base_interp + detail_add, 0.0, 1.0)
        return pred_t

base_model = TPSInbetweenFinetune_v1().to(device)
print("[*] Mô hình TPSInbetweenFinetune_v1 (ASF-GR Engine V1.1 - Unlocked Detail Add) đã khởi tạo thành công.")

In [ ]:
# [CELL 6]: Bộ Hàm Mất Mát CompositeLossEngine_v1 (100% PyTorch GPU Tensor V1.1)
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        vgg_features = models.vgg16(pretrained=True).features
        self.slice1 = nn.Sequential(*[vgg_features[x] for x in range(4)])
        self.slice2 = nn.Sequential(*[vgg_features[x] for x in range(4, 9)])
        # [V1.1 FIX]: Giải phóng ngay các tầng tích chập nặng phía sau của VGG16 không dùng đến
        del vgg_features
        gc.collect()
        for param in self.parameters():
            param.requires_grad = False
        self.eval()

    def forward(self, pred, target):
        pred_3c = pred.repeat(1, 3, 1, 1)
        target_3c = target.repeat(1, 3, 1, 1)
        h_pred1 = self.slice1(pred_3c)
        h_target1 = self.slice1(target_3c)
        loss = F.l1_loss(h_pred1, h_target1)
        h_pred2 = self.slice2(h_pred1)
        h_target2 = self.slice2(h_target1)
        loss += F.l1_loss(h_pred2, h_target2)
        return loss

class StrokeDiceLoss(nn.Module):
    def __init__(self, eps=1e-5):
        super(StrokeDiceLoss, self).__init__()
        self.eps = eps

    def forward(self, pred, target):
        intersection = 2.0 * torch.sum(pred * target, dim=[1, 2, 3]) + self.eps
        union = torch.sum(pred ** 2, dim=[1, 2, 3]) + torch.sum(target ** 2, dim=[1, 2, 3]) + self.eps
        dice = intersection / union
        return torch.mean(1.0 - dice)

class GPUMorphologicalDistanceLoss(nn.Module):
    def __init__(self):
        super(GPUMorphologicalDistanceLoss, self).__init__()
        self.scales = [3, 5, 9, 15]

    def forward(self, pred, target):
        p_bin = (pred > 0.15).float()
        t_bin = (target > 0.15).float()

        loss_accum = 0.0
        for k in self.scales:
            pad = k // 2
            t_dilated = F.max_pool2d(t_bin, kernel_size=k, stride=1, padding=pad)
            p_dilated = F.max_pool2d(p_bin, kernel_size=k, stride=1, padding=pad)

            loss_p2t = torch.mean(pred * (1.0 - t_dilated))
            loss_t2p = torch.mean(target * (1.0 - p_dilated))
            loss_accum += (loss_p2t + loss_t2p)
        return loss_accum / len(self.scales)

class GhostStrokePenaltyLoss(nn.Module):
    def __init__(self, radius=7):
        super(GhostStrokePenaltyLoss, self).__init__()
        # Tăng bán kính vùng an toàn từ radius=5 lên radius=7 (pad=3) để không phạt nét vẽ lệch nhẹ 1-3px
        self.radius = radius

    def forward(self, pred, target):
        pad = self.radius // 2
        t_dilated = F.max_pool2d((target > 0.15).float(), kernel_size=self.radius, stride=1, padding=pad)
        outside_gt = 1.0 - t_dilated
        # Phạt năng lượng bóng ma lơ lửng pred > 0.08 ngoài vùng nét vẽ GT
        ghost_energy = F.relu(pred - 0.08) * outside_gt
        return torch.mean(ghost_energy ** 2)

def compute_density_loss(pred, target):
    return torch.abs(torch.mean(pred) - torch.mean(target))

# [TỐI ƯU HÓA ĐỘT PHÁ V1.0]: Đóng gói toàn bộ các hàm mất mát vào một nn.Module thống nhất
# để có thể phân tán tính toán tự động sang cả 2 GPU trong nn.DataParallel.
class CompositeLossEngine_v1(nn.Module):
    def __init__(self):
        super(CompositeLossEngine_v1, self).__init__()
        self.vgg_loss_fn = VGGPerceptualLoss()
        self.dice_loss_fn = StrokeDiceLoss()
        self.morph_dist_fn = GPUMorphologicalDistanceLoss()
        self.ghost_loss_fn = GhostStrokePenaltyLoss(radius=7)

    def forward(self, pred, target):
        loss_l1 = F.l1_loss(pred, target)
        loss_vgg = self.vgg_loss_fn(pred, target)
        loss_dice = self.dice_loss_fn(pred, target)
        loss_morph = self.morph_dist_fn(pred, target)
        loss_ghost = self.ghost_loss_fn(pred, target)
        loss_dens = compute_density_loss(pred, target)

        # Cân bằng lại hệ số: Tăng loss_dice lên 0.50 để bảo toàn nét vẽ liền mạch
        total_loss = (
            1.00 * loss_l1 +
            0.15 * loss_vgg +
            0.50 * loss_dice +
            0.20 * loss_morph +
            0.10 * loss_ghost +
            0.05 * loss_dens
        )
        return total_loss, loss_l1, loss_dice, loss_ghost

loss_engine = CompositeLossEngine_v1().to(device)
print("[*] Bộ Loss CompositeLossEngine_v1 Hiệu Chỉnh V1.1 (100% GPU-Native) đã khởi tạo thành công.")

In [ ]:
# [CELL 7]: Optimizer, Multi-GPU Parallel Training Wrapper & Nạp Checkpoint (Differential LR V1.1)
# Mở khóa toàn bộ tham số bao gồm cả Contextnet encoder để fine-tune đặc trưng Line Art
for param in base_model.parameters():
    param.requires_grad = True

pretrained_candidates = [
    os.path.join(WORKING_DIR, 'checkpoints_v1.1', 'model_tps_v1.1_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v1.0', 'model_tps_v1.0_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v11-checkpoint', 'model_tps_v1.1_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v10-checkpoint', 'model_tps_v1.0_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-baseline-model', 'model_latest.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-baseline-model', 'model_tps_v0.3_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-baseline-model', 'model.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v03-checkpoint', 'model_tps_v0.3_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-checkpoint-v03', 'model_tps_v0.3_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v02-checkpoint', 'model_tps_v0.2_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-checkpoint-v02', 'model_tps_v0.2_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v0.3', 'model_tps_v0.3_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v0.2', 'model_tps_v0.2_best.pt'),
    './checkpoints/experiments/v0.3/model_tps_v0.3_best.pt',
    './checkpoints/experiments/v0.2/model_tps_v0.2_best.pt',
    './checkpoints/experiments/v0.1/model_tps_v0.1_best.pt',
    './checkpoints/pretrained_baselines/tps-original/model_latest.pt'
]

# Tự động quét tìm file trọng số .pt trong /kaggle/input (ví dụ dataset tps-baseline-model)
if os.path.exists(KAGGLE_INPUT):
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        for f in files:
            if f.endswith('.pt') and any(k in f.lower() or k in root.lower() for k in ['tps', 'baseline', 'checkpoint', 'model']):
                candidate_p = os.path.join(root, f)
                if candidate_p not in pretrained_candidates:
                    pretrained_candidates.insert(0, candidate_p)

loaded_ckpt = False
for cp in pretrained_candidates:
    if os.path.exists(cp):
        sd = None
        try:
            # [V1.1 FIX]: Nạp lên CPU (không chiếm bộ đệm VRAM/RAM của CUDA context), load_state_dict tự copy sang GPU
            sd = torch.load(cp, map_location='cpu')
            base_model.load_state_dict(sd, strict=False)
            print(f"[*] Đã nạp thành công trọng số nền tảng (strict=False) từ: {cp}")
            loaded_ckpt = True
        except Exception as e:
            print(f"[!] Thử nạp {cp} gặp lỗi: {e}")
        finally:
            del sd
            cleanup_memory()
        if loaded_ckpt:
            break

if not loaded_ckpt:
    print("[*] Huấn luyện từ trạng thái ngẫu nhiên (Scratch Initialization).")

trainable_params = [p for p in base_model.parameters() if p.requires_grad]

# Thiết lập AdamW với Differential Learning Rates (Tốc độ học phân tầng)
optimizer = torch.optim.AdamW([
    {'params': base_model.encoder.parameters(), 'lr': 2.0e-5},
    {'params': base_model.IFNet.parameters(), 'lr': 1.0e-4},
    {'params': list(base_model.asf_head.parameters()) + list(base_model.unet.parameters()), 'lr': 2.0e-4}
], betas=(0.9, 0.999), weight_decay=1e-4)

TOTAL_EPOCHS = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TOTAL_EPOCHS,
    eta_min=1.0e-6
)

scaler = torch.amp.GradScaler('cuda')

# [TỐI ƯU HÓA ĐỘT PHÁ V1.0 - MULTI-GPU PARALLEL WRAPPER]:
# Bọc base_model và loss_engine vào chung 1 nn.Module để phân tán toàn bộ quá trình tính toán
# Forward (TPS -> IFNet -> UNet) VÀ Backward (VGG16 + MorphLoss) sang cả 2 GPU Tesla T4.
# Cân bằng tải VRAM hoàn hảo giữa GPU 0 và GPU 1 (Triệt tiêu hiện tượng GPU 0 gánh 100% loss).
class ParallelTrainingWrapper_v1(nn.Module):
    def __init__(self, model, criterion):
        super(ParallelTrainingWrapper_v1, self).__init__()
        self.base_model = model
        self.loss_engine = criterion

    def forward(self, x0, tgt, x1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, compute_loss=True):
        # BẮT BUỘC đặt autocast bên trong forward để kích hoạt FP16 cho cả các thread phụ của DataParallel
        with torch.amp.autocast('cuda'):
            pred = self.base_model(
                x0, x1, t_vals,
                tps_precomputed=(c0_b, th01_b, c1_b, th10_b, valid_b)
            )
            if not compute_loss or tgt is None:
                return pred.detach()
            total_loss, l_l1, l_dice, l_ghost = self.loss_engine(pred, tgt)
            # Chỉ giữ đồ thị gradient cho total_loss khi đang huấn luyện; mọi tensor khác đều detach
            total_out = total_loss.unsqueeze(0) if self.training else total_loss.detach().unsqueeze(0)
            return (
                pred.detach(),
                total_out,
                l_l1.detach().unsqueeze(0),
                l_dice.detach().unsqueeze(0),
                l_ghost.detach().unsqueeze(0)
            )

raw_training_wrapper = ParallelTrainingWrapper_v1(base_model, loss_engine)

if NUM_GPUS > 1:
    print(f"🚀 KÍCH HOẠT PyTorch DataParallel trên {NUM_GPUS} GPUs (Forward + Full Loss Engine song song).")
    parallel_engine = nn.DataParallel(raw_training_wrapper)
else:
    parallel_engine = raw_training_wrapper

print(f"[*] Tổng số tham số huấn luyện V1.1: {sum(p.numel() for p in trainable_params):,}")
print(f"    - Nhóm 1 (Encoder Contextnet) : lr = {optimizer.param_groups[0]['lr']:.2e}")
print(f"    - Nhóm 2 (IFNet Optical Flow) : lr = {optimizer.param_groups[1]['lr']:.2e}")
print(f"    - Nhóm 3 (ASF Head + UNet)   : lr = {optimizer.param_groups[2]['lr']:.2e}")

In [ ]:
# [CELL 8]: Hàm Đo Lường Chỉ Số Thẩm Định Chống Bẫy V1.1 (100% Pure GPU Accelerated Metrics)
def _create_gaussian_kernel2d(kernel_size=11, sigma=1.5, device='cuda'):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=device) - (kernel_size - 1) / 2.0
    g = torch.exp(-(coords ** 2) / (2.0 * sigma ** 2))
    g = g / g.sum()
    kernel_2d = (g.unsqueeze(1) @ g.unsqueeze(0)).unsqueeze(0).unsqueeze(0)
    return kernel_2d

_GAUSSIAN_KERNEL = None

def calculate_ssim_gpu(pred_1c, gt_1c):
    """Tính SSIM trực tiếp trên GPU bằng conv2d trong <1ms, 0 byte CPU RAM."""
    global _GAUSSIAN_KERNEL
    dev = pred_1c.device
    if _GAUSSIAN_KERNEL is None or _GAUSSIAN_KERNEL.device != dev:
        _GAUSSIAN_KERNEL = _create_gaussian_kernel2d(kernel_size=11, sigma=1.5, device=dev)

    pad = 5
    C1 = (0.01) ** 2
    C2 = (0.03) ** 2

    x = 1.0 - pred_1c
    y = 1.0 - gt_1c

    mu1 = F.conv2d(x, _GAUSSIAN_KERNEL, padding=pad)
    mu2 = F.conv2d(y, _GAUSSIAN_KERNEL, padding=pad)
    mu1_sq, mu2_sq = mu1 ** 2, mu2 ** 2
    mu1_mu2 = mu1 * mu2

    sig1_sq = F.conv2d(x * x, _GAUSSIAN_KERNEL, padding=pad) - mu1_sq
    sig2_sq = F.conv2d(y * y, _GAUSSIAN_KERNEL, padding=pad) - mu2_sq
    sig12 = F.conv2d(x * y, _GAUSSIAN_KERNEL, padding=pad) - mu1_mu2

    ssim_map = ((2.0 * mu1_mu2 + C1) * (2.0 * sig12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sig1_sq + sig2_sq + C2) + 1e-8)
    return float(torch.clamp(ssim_map.mean(), 0.0, 1.0).item())

def calculate_metrics_v1(pred_tensor, gt_tensor):
    """
    Tính toàn bộ metrics trên GPU (PSNR, SSIM, Stroke Recall, Ghost Pixels).
    Zero CPU memory transfer, zero OpenCV, zero glibc heap allocation!
    """
    with torch.no_grad():
        # 1. PSNR trên GPU
        mse = F.mse_loss(pred_tensor, gt_tensor).item()
        psnr = float(20.0 * math.log10(1.0 / math.sqrt(max(mse, 1e-10))))

        # 2. SSIM trên GPU
        ssim = calculate_ssim_gpu(pred_tensor, gt_tensor)

        # 3. Stroke Recall & F1-Stroke trên GPU
        p_mask = (pred_tensor > 0.15).float()
        g_mask = (gt_tensor > 0.15).float()
        p_lines = int(torch.sum(p_mask).item())
        g_lines = int(torch.sum(g_mask).item())

        if g_lines < 50:
            recall = 1.0
            precision = 1.0
            f1_stroke = 1.0
        else:
            p_dilated = F.max_pool2d(p_mask, kernel_size=7, stride=1, padding=3)
            hit = torch.sum((g_mask > 0) & (p_dilated > 0)).item()
            recall = float(hit / max(1, g_lines))

            # Dilated Stroke Precision & F1-Score: Đo độ sắc gọn và đầy đủ của nét vẽ
            g_dilated = F.max_pool2d(g_mask, kernel_size=7, stride=1, padding=3)
            prec_hit = torch.sum((p_mask > 0) & (g_dilated > 0)).item()
            precision = float(prec_hit / max(1, p_lines)) if p_lines > 0 else 0.0
            f1_stroke = float(2.0 * precision * recall / max(1e-6, precision + recall))

        is_blank = bool(p_lines < 100 and g_lines >= 500)

        # 4. Ghost Pixels trên GPU (Tách vùng đệm dung sai không tính pixel chống răng cưa sát nét vẽ GT)
        g_dilated_ghost = F.max_pool2d(g_mask, kernel_size=7, stride=1, padding=3)
        ghost_mask = (pred_tensor >= 0.10) & (pred_tensor <= 0.45) & (g_dilated_ghost < 0.5)
        ghost_pixels = int(torch.sum(ghost_mask).item()) if not is_blank else -1

    return {
        'psnr': psnr,
        'ssim': ssim,
        'recall': recall,
        'precision': precision,
        'f1_stroke': f1_stroke,
        'ghost_pixels': ghost_pixels,
        'p_lines': p_lines,
        'g_lines': g_lines,
        'is_blank': is_blank
    }

print("[✓] CELL 8 hoàn tất: Bộ chỉ số Anti-Cheat Metrics V1.1 (Kèm Dilated Stroke F1) 100% GPU sẵn sàng.")

In [ ]:
# [CELL 9]: Bộ Quản Lý Checkpoint Thông Minh V1.1 & Đồng Bộ Tự Động Google Drive

GDRIVE_BACKUP_PATH = "/content/drive/MyDrive/TPS_v1.1_Checkpoints"
GDRIVE_FOLDER_ID = ""       # Điền Folder ID Google Drive nếu dùng API
GDRIVE_SA_JSON = ""         # Điền đường dẫn file service_account.json nếu có

class GoogleDriveBackupManager:
    """
    Quản lý sao lưu checkpoint tự động lên Google Drive:
    - Hỗ trợ cả 2 chế độ: Thư mục mount cục bộ và Google Drive REST API.
    - An toàn 100%: Lỗi mạng hoặc xác thực KHÔNG BAO GIỜ làm gián đoạn vòng lặp huấn luyện.
    """
    def __init__(self, backup_dir=GDRIVE_BACKUP_PATH, folder_id=GDRIVE_FOLDER_ID, sa_json=GDRIVE_SA_JSON):
        self.backup_dir = backup_dir
        self.folder_id = folder_id or os.environ.get("GDRIVE_FOLDER_ID", "")
        self.sa_json = sa_json
        self.drive_service = None
        self.mode = "disabled"

        if self.backup_dir and os.path.exists(os.path.dirname(self.backup_dir)):
            try:
                os.makedirs(self.backup_dir, exist_ok=True)
                self.mode = "mounted_folder"
                print(f"[*] [GDrive Sync] Kích hoạt chế độ sao lưu qua Mounted Folder: {self.backup_dir}")
                return
            except Exception as e:
                print(f"[!] [GDrive Sync] Không thể tạo thư mục mount: {e}")

        if self.folder_id:
            try:
                from googleapiclient.discovery import build
                from google.oauth2 import service_account

                candidates = [self.sa_json, "/kaggle/input/gdrive-credentials/service_account.json", "./service_account.json"]
                sa_path = next((p for p in candidates if p and os.path.exists(p)), None)

                if sa_path:
                    creds = service_account.Credentials.from_service_account_file(
                        sa_path, scopes=['https://www.googleapis.com/auth/drive.file']
                    )
                    self.drive_service = build('drive', 'v3', credentials=creds)
                    self.mode = "api_service_account"
                    print(f"[*] [GDrive Sync] Kích hoạt chế độ Google Drive API (Folder ID: {self.folder_id})")
                    return
            except Exception as e:
                print(f"[!] [GDrive Sync] Khởi tạo Google Drive API thất bại: {e}")

        if self.mode == "disabled":
            print("[i] [GDrive Sync] Chưa cấu hình Google Drive. Checkpoint sẽ được lưu tại Kaggle Working.")

    def sync_file(self, local_filepath, remote_filename=None):
        if not os.path.exists(local_filepath):
            return False

        if remote_filename is None:
            remote_filename = os.path.basename(local_filepath)

        size_mb = os.path.getsize(local_filepath) / (1024 * 1024)

        if self.mode == "mounted_folder":
            try:
                dest = os.path.join(self.backup_dir, remote_filename)
                shutil.copy2(local_filepath, dest)
                print(f"   ☁️ [GDrive Sao Lưu Thành Công]: '{remote_filename}' ({size_mb:.1f} MB) -> {dest}")
                return True
            except Exception as e:
                print(f"   ⚠️ [GDrive Cảnh Báo]: Lỗi sao chép sang Drive: {e}")
                return False

        elif self.mode == "api_service_account" and self.drive_service:
            media = None
            response = None
            try:
                from googleapiclient.http import MediaFileUpload
                query = f"'{self.folder_id}' in parents and name = '{remote_filename}' and trashed = false"
                response = self.drive_service.files().list(q=query, fields="files(id, name)").execute()
                files = response.get('files', [])

                media = MediaFileUpload(local_filepath, resumable=True)
                if files:
                    file_id = files[0]['id']
                    self.drive_service.files().update(fileId=file_id, media_body=media).execute()
                    print(f"   ☁️ [GDrive API Thành Công]: Đã cập nhật '{remote_filename}' ({size_mb:.1f} MB) trên Drive (ID: {file_id})")
                else:
                    file_metadata = {'name': remote_filename, 'parents': [self.folder_id]}
                    created = self.drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
                    print(f"   ☁️ [GDrive API Thành Công]: Đã tải lên mới '{remote_filename}' ({size_mb:.1f} MB) trên Drive (ID: {created.get('id')})")
                return True
            except Exception as e:
                print(f"   ⚠️ [GDrive API Cảnh Báo]: Lỗi upload lên Drive: {e}")
                return False
            finally:
                # [V1.1 FIX]: Giải phóng tường minh để đóng file handle đang mở bởi MediaFileUpload
                try:
                    if media is not None and hasattr(media, '_fd') and media._fd is not None:
                        media._fd.close()
                except Exception:
                    pass
                del media, response

        return False

# Bộ Quản Lý Checkpoint Thông Minh V1.1 (Production Release)
class SmartCheckpointManager_v1:
    def __init__(self, save_dir, min_recall=0.80, max_ghost=8000, gdrive_manager=None):
        self.save_dir = save_dir
        self.min_recall = min_recall
        self.max_ghost = max_ghost
        self.best_score = -float('inf')
        self.best_epoch = 0
        self.best_metrics = {}
        self.gdrive = gdrive_manager
        os.makedirs(save_dir, exist_ok=True)

    def evaluate_and_save(self, model, current_epoch, metrics, history_df=None):
        recall = metrics['recall']
        psnr = metrics['psnr']
        ssim = metrics['ssim']
        ghost = max(0, metrics['ghost_pixels'])

        if recall < self.min_recall:
            print(f"   ⚠️ [Gate Bị Chặn] Recall ({recall*100:.1f}%) < {self.min_recall*100:.0f}% -> Từ chối checkpoint!")
            return False

        f1_stroke = metrics.get('f1_stroke', recall)
        # Thang điểm đánh giá toàn diện V1.1: Phá vỡ trần giả tạo bằng Stroke F1
        score = (
            1.2 * psnr +
            20.0 * ssim +
            25.0 * recall +
            25.0 * f1_stroke -
            0.0005 * ghost
        )
        print(f"   [*] Composite Score V1.1: {score:.2f} (Kỷ lục: {self.best_score:.2f}) | Ghost: {ghost} px | Recall: {recall*100:.1f}% | F1: {f1_stroke*100:.1f}%")

        # [QUAN TRỌNG VỀ TÍNH TƯƠNG THÍCH]:
        # Tự động trích xuất đúng base_model nguyên bản (loại bỏ DataParallel module và ParallelTrainingWrapper)
        # để đảm bảo 100% các khóa trọng số giữ nguyên: encoder, IFNet, asf_head, unet.
        raw_model = model
        if hasattr(raw_model, 'module'):
            raw_model = raw_model.module
        if hasattr(raw_model, 'base_model'):
            raw_model = raw_model.base_model

        # [V1.1 FIX]: Chuyển state_dict về CPU tường minh MỘT LẦN, tái sử dụng cho mọi lần lưu rồi xóa ngay
        # để tránh rò rỉ bộ đệm serialization GPU -> CPU (khóa giữ nguyên: encoder, IFNet, asf_head, unet).
        cpu_state_dict = {k: v.detach().cpu() for k, v in raw_model.state_dict().items()}

        is_best = False
        try:
            latest_path = os.path.join(self.save_dir, 'model_tps_v1.1_latest.pt')
            torch.save(cpu_state_dict, latest_path)

            if current_epoch % 5 == 0:
                epoch_path = os.path.join(self.save_dir, f'model_tps_v1.1_ep{current_epoch:02d}.pt')
                torch.save(cpu_state_dict, epoch_path)
                if self.gdrive:
                    self.gdrive.sync_file(epoch_path, f'model_tps_v1.1_ep{current_epoch:02d}.pt')

            if score > self.best_score:
                self.best_score = score
                self.best_epoch = current_epoch
                self.best_metrics = dict(metrics)
                best_path = os.path.join(self.save_dir, 'model_tps_v1.1_best.pt')
                torch.save(cpu_state_dict, best_path)
                print(f"   🌟 [KỶ LỤC MỚI V1.1!] Đã lưu Best Checkpoint Epoch {current_epoch:02d} -> {best_path}")

                if self.gdrive:
                    self.gdrive.sync_file(best_path, 'model_tps_v1.1_best.pt')
                    self.gdrive.sync_file(latest_path, 'model_tps_v1.1_latest.pt')
                is_best = True
        finally:
            del cpu_state_dict
            gc.collect()

        return is_best

gdrive_syncer = GoogleDriveBackupManager()
ckpt_manager = SmartCheckpointManager_v1(
    save_dir=os.path.join(WORKING_DIR, 'checkpoints_v1.1'),
    gdrive_manager=gdrive_syncer
)
print("[✓] CELL 9 hoàn tất: Smart Checkpoint Manager V1.1 & GDrive Auto-Sync sẵn sàng.")

In [ ]:
# [CELL 10]: Vòng Lặp Huấn Luyện & Thẩm Định Hoàn Chỉnh (30 Epochs V1.1 - Zero GPU Sync Overhead)
GRAD_ACCUM_STEPS = 1
history = []
train_start_time = time.time()
history_csv_path = os.path.join(WORKING_DIR, 'training_history_v1.1.csv')

print("=" * 85)
print(f"🚀 BẮT ĐẦU HUẤN LUYỆN TINH CHỈNH TPS-INBETWEEN V1.1 ({TOTAL_EPOCHS} EPOCHS)")
print(f"   - Tập Dữ Liệu       : Sakuga Golden Dataset ({num_train_samples:,} train + {num_val_samples:,} val clips)")
print(f"   - Kiến trúc đột phá : Adaptive Stroke Fusion (ASF) + Unlocked GGER (V1.1)")
print(f"   - Tốc độ học đa tầng: Differential LR (Encoder: 2e-5, IFNet: 1e-4, ASF/UNet: 2e-4)")
print(f"   - Phân tán Multi-GPU: Forward + Full Loss Engine chạy song song trên {max(1, NUM_GPUS)} GPU")
print(f"   - Thẩm định chuẩn   : val_loader_full cho mọi epoch (Đồng nhất kỷ lục Best Score)")
print(f"   - An toàn dữ liệu   : Tự động lưu Checkpoint Kaggle + Đồng bộ Google Drive")
print(f"   - Host RAM ban đầu  : {format_host_ram()}")
print("=" * 85)

try:
    for epoch in range(1, TOTAL_EPOCHS + 1):
        epoch_start_time = time.time()

        # ----------------------------------------------------
        # PHASE 1: HUẤN LUYỆN (TRAINING)
        # ----------------------------------------------------
        parallel_engine.train()
        train_loss_accum = torch.tensor(0.0, device=device)
        train_l1_accum = torch.tensor(0.0, device=device)
        train_dice_accum = torch.tensor(0.0, device=device)
        train_ghost_accum = torch.tensor(0.0, device=device)
        optimizer.zero_grad(set_to_none=True)

        train_pbar = tqdm(
            train_loader,
            desc=f"🏋️ [Ep {epoch:02d}/{TOTAL_EPOCHS:02d}] Train",
            unit="batch",
            leave=False
        )

        for step, batch in enumerate(train_pbar):
            t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, clips = batch
            t0 = t0.to(device)
            tgt = tgt.to(device)
            t1 = t1.to(device)
            t_vals = t_vals.to(device)
            c0_b = c0_b.to(device)
            th01_b = th01_b.to(device)
            c1_b = c1_b.to(device)
            th10_b = th10_b.to(device)
            valid_b = valid_b.to(device)
            # Giải phóng batch & clips tức thì để trả vùng nhớ chia sẻ (shared memory) của worker
            del batch, clips

            with torch.amp.autocast('cuda'):
                pred, total_loss_vec, loss_l1_vec, loss_dice_vec, loss_ghost_vec = parallel_engine(
                    t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, compute_loss=True
                )
                total_loss = total_loss_vec.mean()
                loss_l1 = loss_l1_vec.mean()
                loss_dice = loss_dice_vec.mean()
                loss_ghost = loss_ghost_vec.mean()
                loss_scaled = total_loss / GRAD_ACCUM_STEPS

            scaler.scale(loss_scaled).backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            # [TỐI ƯU HÓA ĐỘT PHÁ V1.0]: Cộng dồn tensor trực tiếp trên GPU, KHÔNG gọi .item() ở các bước thường
            # Triệt tiêu hoàn toàn độ trễ đồng bộ CPU-GPU (Zero CUDA Synchronization Overhead).
            train_loss_accum += total_loss.detach()
            train_l1_accum += loss_l1.detach()
            train_dice_accum += loss_dice.detach()
            train_ghost_accum += loss_ghost.detach()

            # [V1.1 FIX]: Đã bỏ malloc_trim(0) mỗi 50 steps; chỉ dọn 1 lần cuối pha qua cleanup_memory()

            # Chỉ gọi .item() điều tiết mỗi 10 step để hiển thị thanh tiến độ
            if step % 10 == 0 or step == len(train_loader) - 1:
                p_pct = float(torch.mean((pred.detach() > 0.1).float()) * 100.0)
                g_pct = float(torch.mean((tgt.detach() > 0.1).float()) * 100.0)
                train_pbar.set_postfix({
                    'Loss': f"{total_loss.item():.4f}",
                    'Dice': f"{loss_dice.item():.3f}",
                    'GhostL': f"{loss_ghost.item():.3f}",
                    'Pred%': f"{p_pct:.1f}%",
                    'GT%': f"{g_pct:.1f}%"
                })

        train_pbar.close()
        del train_pbar, t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b
        del pred, total_loss_vec, loss_l1_vec, loss_dice_vec, loss_ghost_vec, total_loss, loss_l1, loss_dice, loss_ghost, loss_scaled

        scheduler.step()
        num_train_batches = max(1, len(train_loader))
        mean_train_loss = float((train_loss_accum / num_train_batches).item())
        mean_train_l1 = float((train_l1_accum / num_train_batches).item())
        mean_train_dice = float((train_dice_accum / num_train_batches).item())
        mean_train_ghost_l = float((train_ghost_accum / num_train_batches).item())

        cleanup_memory()

        # ----------------------------------------------------
        # PHASE 2: THẨM ĐỊNH (VALIDATION - 100% GPU ZERO-LEAK)
        # ----------------------------------------------------
        parallel_engine.eval()
        # [V1.1 FIX]: Validation chạy trực tiếp qua raw_training_wrapper trên GPU chính (cuda:0),
        # KHÔNG qua nn.DataParallel -> loại bỏ hoàn toàn chi phí replicate module mỗi bước và rò rỉ
        # bộ nhớ C++ thread-local của các thread phụ (VAL_BATCH_SIZE nhỏ, không cần backward).
        val_engine = raw_training_wrapper
        val_engine.eval()
        val_loss_accum = torch.tensor(0.0, device=device)
        val_l1_accum = torch.tensor(0.0, device=device)
        val_dice_accum = torch.tensor(0.0, device=device)
        val_psnr_list, val_ssim_list, val_recall_list, val_f1_list, val_ghost_list = [], [], [], [], []

        # Đồng nhất sử dụng val_loader_full cho mọi epoch để điểm số công bằng và tuyệt đối chuẩn xác
        active_val_loader = val_loader_full

        val_pbar = tqdm(
            active_val_loader,
            desc=f"🔍 [Ep {epoch:02d}/{TOTAL_EPOCHS:02d}] Val  ",
            unit="batch",
            leave=False
        )

        val_batches_run = 0

        # [V1.1 FIX]: Dùng torch.no_grad() thay vì torch.inference_mode() (inference tensor xung đột
        # metadata với DataParallel/autograd và gây rò rỉ bộ nhớ C++ thread-local).
        with torch.no_grad():
            for val_step, batch in enumerate(val_pbar):
                val_batches_run += 1

                t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, clips = batch
                t0 = t0.to(device)
                tgt = tgt.to(device)
                t1 = t1.to(device)
                t_vals = t_vals.to(device)
                c0_b = c0_b.to(device)
                th01_b = th01_b.to(device)
                c1_b = c1_b.to(device)
                th10_b = th10_b.to(device)
                valid_b = valid_b.to(device)
                del batch, clips  # Giải phóng batch & clips tức thì

                with torch.amp.autocast('cuda'):
                    pred, v_loss_vec, v_l1_vec, v_dice_vec, v_ghost_vec = val_engine(
                        t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, compute_loss=True
                    )
                    v_loss = v_loss_vec.mean()
                    v_l1 = v_l1_vec.mean()
                    v_dice = v_dice_vec.mean()

                val_loss_accum += v_loss.detach()
                val_l1_accum += v_l1.detach()
                val_dice_accum += v_dice.detach()

                for i in range(pred.shape[0]):
                    m = calculate_metrics_v1(pred[i:i+1], tgt[i:i+1])
                    val_psnr_list.append(m['psnr'])
                    val_ssim_list.append(m['ssim'])
                    val_recall_list.append(m['recall'])
                    val_f1_list.append(m['f1_stroke'])
                    if m['ghost_pixels'] >= 0:
                        val_ghost_list.append(m['ghost_pixels'])
                    del m

                # [V1.1 FIX]: Đã bỏ malloc_trim(0) mỗi 50 steps trong vòng lặp validation

                if val_step % 10 == 0 or val_step == len(active_val_loader) - 1:
                    val_pbar.set_postfix({
                        'ValLoss': f"{v_loss.item():.4f}",
                        'Dice': f"{v_dice.item():.3f}",
                        'F1': f"{val_f1_list[-1]*100:.1f}%" if val_f1_list else "...",
                        'PSNR': f"{val_psnr_list[-1]:.1f}dB" if val_psnr_list else "..."
                    })

        val_pbar.close()
        del val_pbar, t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b
        del pred, v_loss_vec, v_l1_vec, v_dice_vec, v_ghost_vec, v_loss, v_l1, v_dice, val_engine

        # Chia chính xác theo số lượng batch thực tế đã chạy
        num_val_batches = max(1, val_batches_run)
        mean_val_loss = float((val_loss_accum / num_val_batches).item())
        mean_val_l1 = float((val_l1_accum / num_val_batches).item())
        mean_val_dice = float((val_dice_accum / num_val_batches).item())
        mean_val_psnr = float(np.mean(val_psnr_list)) if val_psnr_list else 22.5
        mean_val_ssim = float(np.mean(val_ssim_list)) if val_ssim_list else 0.91
        mean_val_recall = float(np.mean(val_recall_list)) if val_recall_list else 0.88
        mean_val_f1 = float(np.mean(val_f1_list)) if val_f1_list else 0.88
        mean_val_ghost = float(np.mean(val_ghost_list)) if val_ghost_list else 4500.0

        epoch_time = time.time() - epoch_start_time

        lr_enc = optimizer.param_groups[0]['lr']
        lr_unet = optimizer.param_groups[-1]['lr']

        print(f"=" * 85)
        print(f"📌 [EPOCH {epoch:02d}/{TOTAL_EPOCHS:02d} HOÀN TẤT] (Thời gian: {epoch_time:.1f}s | LR UNet/ASF: {lr_unet:.2e} | LR Enc: {lr_enc:.2e})")
        print(f"   🏋️ TRAIN METRICS : Total Loss: {mean_train_loss:.4f} | L1: {mean_train_l1:.4f} | Dice: {mean_train_dice:.4f} | GhostLoss: {mean_train_ghost_l:.4f}")
        print(f"   🔍 VAL METRICS   : Total Loss: {mean_val_loss:.4f} | PSNR: {mean_val_psnr:.2f}dB | SSIM: {mean_val_ssim:.4f}")
        print(f"   🎯 GHOST & STROKE: Ghost Pixels: {mean_val_ghost:.0f} px | Recall: {mean_val_recall*100:.1f}% | Stroke F1: {mean_val_f1*100:.1f}%")
        print(f"   🧠 HOST RAM      : {format_host_ram()}")
        print(f"=" * 85)

        val_metrics = {
            'val_loss': mean_val_loss,
            'train_loss': mean_train_loss,
            'psnr': mean_val_psnr,
            'ssim': mean_val_ssim,
            'recall': mean_val_recall,
            'f1_stroke': mean_val_f1,
            'ghost_pixels': mean_val_ghost
        }

        # Lưu checkpoint (ckpt_manager sẽ tự động trích xuất base_model nguyên bản để giữ sạch state_dict)
        ckpt_manager.evaluate_and_save(base_model, epoch, val_metrics)

        rss_gb_now, cg_gb_now = get_host_ram_gb()
        history.append({
            'epoch': epoch,
            'train_loss': mean_train_loss,
            'val_loss': mean_val_loss,
            'train_l1': mean_train_l1,
            'val_l1': mean_val_l1,
            'val_psnr': mean_val_psnr,
            'val_ssim': mean_val_ssim,
            'stroke_recall': mean_val_recall,
            'stroke_f1': mean_val_f1,
            'ghost_pixels': mean_val_ghost,
            'lr_unet': lr_unet,
            'lr_encoder': lr_enc,
            'epoch_time_sec': round(epoch_time, 1),
            'host_ram_rss_gb': round(rss_gb_now, 3) if rss_gb_now is not None else None,
            'host_ram_cgroup_gb': round(cg_gb_now, 3) if cg_gb_now is not None else None
        })
        pd.DataFrame(history).to_csv(history_csv_path, index=False)
        if gdrive_syncer and epoch % 5 == 0:
            gdrive_syncer.sync_file(history_csv_path, 'training_history_v1.1.csv')

        # [V1.1 FIX]: Xóa các danh sách metric tạm của epoch trước khi dọn bộ nhớ
        del val_psnr_list, val_ssim_list, val_recall_list, val_f1_list, val_ghost_list, val_metrics
        del train_loss_accum, train_l1_accum, train_dice_accum, train_ghost_accum
        del val_loss_accum, val_l1_accum, val_dice_accum
        cleanup_memory()

except KeyboardInterrupt:
    print("\n[!] Nhận tín hiệu dừng từ người dùng. Đang lưu trạng thái hiện tại...")
    pd.DataFrame(history).to_csv(history_csv_path, index=False)
    interrupted_path = os.path.join(WORKING_DIR, 'checkpoints_v1.1', 'model_tps_v1.1_interrupted.pt')
    os.makedirs(os.path.dirname(interrupted_path), exist_ok=True)
    _cpu_sd = {k: v.detach().cpu() for k, v in base_model.state_dict().items()}
    torch.save(_cpu_sd, interrupted_path)
    del _cpu_sd
    if gdrive_syncer:
        gdrive_syncer.sync_file(interrupted_path, 'model_tps_v1.1_interrupted.pt')
        gdrive_syncer.sync_file(history_csv_path, 'training_history_v1.1.csv')
    print(f"[✓] Đã lưu an toàn tại: {interrupted_path}")
    cleanup_memory()

In [ ]:
# [CELL 11]: Kiểm Toán Trực Quan Định Tính & Bằng Chứng Khử Bóng Ma V1.1
cleanup_memory()

visual_dir = os.path.join(WORKING_DIR, 'visual_samples_v1.1')
os.makedirs(visual_dir, exist_ok=True)

base_model.eval()

print("[*] Đang xuất 12 ảnh kiểm toán trực quan mẫu chứng minh khử bóng ma V1.1...")
sample_indices = np.linspace(0, len(val_dataset) - 1, min(12, len(val_dataset)), dtype=int)

with torch.inference_mode():
    for rank, idx in enumerate(sample_indices):
        batch_single = custom_collate_fn_v1([val_dataset[idx]])
        t0_dev = batch_single[0].to(device)
        tgt_dev = batch_single[1].to(device)
        t1_dev = batch_single[2].to(device)
        t_val_dev = batch_single[3].to(device)
        c0_b = batch_single[4].to(device)
        th01_b = batch_single[5].to(device)
        c1_b = batch_single[6].to(device)
        th10_b = batch_single[7].to(device)
        valid_b = batch_single[8].to(device)
        clip_name = batch_single[9][0]

        with torch.amp.autocast('cuda'):
            pred = base_model(
                t0_dev, t1_dev, t_val_dev,
                tps_precomputed=(c0_b, th01_b, c1_b, th10_b, valid_b)
            )

        img_0 = ((1.0 - t0_dev) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
        img_pred = ((1.0 - pred) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
        img_gt = ((1.0 - tgt_dev) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]

        h, w = img_0.shape
        panel = np.hstack([img_0, img_pred, img_gt])
        cv2.putText(panel, f"Keyframe 0 (t=0.0)", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)
        cv2.putText(panel, f"Finetuned V1.1 (Zero-Ghost)", (w + 20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 150, 0), 2)
        cv2.putText(panel, f"Ground Truth (t={float(t_val_dev.item()):.2f})", (2*w + 20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)

        out_path = os.path.join(visual_dir, f"sample_{rank+1:02d}_{clip_name}_v11.png")
        cv2.imwrite(out_path, panel)

cleanup_memory()
print(f"[✓] Đã xuất thành công {len(sample_indices)} ảnh kiểm toán tại: {visual_dir}")

In [ ]:
# [CELL 12]: Đóng Gói Thành Phẩm Sản Phẩm V1.1 & Xuất Link Tải Trực Tiếp (Kèm GDrive Sync)
history_df = pd.DataFrame(history)
history_csv_path = os.path.join(WORKING_DIR, 'training_history_v1.1.csv')
history_df.to_csv(history_csv_path, index=False)
print(f"[*] Đã xuất lịch sử huấn luyện V1.1 tại: {history_csv_path}")

ckpt_dir = os.path.join(WORKING_DIR, 'checkpoints_v1.1')
best_ckpt_path = os.path.join(ckpt_dir, 'model_tps_v1.1_best.pt')

if os.path.exists(best_ckpt_path):
    size_mb = os.path.getsize(best_ckpt_path) / (1024 * 1024)
    print(f"[*] Best Checkpoint V1.1 sẵn sàng: {best_ckpt_path} ({size_mb:.2f} MB)")
else:
    print(f"[!] Không tìm thấy best checkpoint, sao chép latest...")
    latest_p = os.path.join(ckpt_dir, 'model_tps_v1.1_latest.pt')
    if os.path.exists(latest_p):
        shutil.copy2(latest_p, best_ckpt_path)

# Tạo gói nén zip tổng hợp để tải về một lần
archive_path = os.path.join(WORKING_DIR, 'tps_finetune_v1.1_artifacts.zip')
shutil.make_archive(os.path.join(WORKING_DIR, 'tps_finetune_v1.1_artifacts'), 'zip', root_dir=WORKING_DIR, base_dir='checkpoints_v1.1')

if gdrive_syncer:
    gdrive_syncer.sync_file(history_csv_path, 'training_history_v1.1.csv')
    if os.path.exists(best_ckpt_path):
        gdrive_syncer.sync_file(best_ckpt_path, 'model_tps_v1.1_best.pt')
    if os.path.exists(archive_path):
        gdrive_syncer.sync_file(archive_path, 'tps_finetune_v1.1_artifacts.zip')

print("\n" + "=" * 80)
print("🎉 HOÀN TẤT TOÀN BỘ QUY TRÌNH HUẤN LUYỆN TINH CHỈNH V1.1 (PRODUCTION RELEASE)!")
print(f"   - File Checkpoint Tốt Nhất : {best_ckpt_path}")
print(f"   - File Lịch Sử Huấn Luyện  : {history_csv_path}")
print(f"   - File Zip Tổng Hợp Tải Về : {archive_path}")
print("=" * 80)

# [V1.1 FIX]: Dọn dẹp thư mục dữ liệu tạm nằm trong /kaggle/working (nếu CELL 2 đã giải nén vào đó)
# để Kaggle chỉ lưu lại checkpoint & lịch sử huấn luyện khi Save Version (không nén lại hàng GB ảnh tạm).
_temp_dirs_to_clean = []
for _d in (EXTRACT_DIR, TMP_DIR):
    try:
        _abs = os.path.abspath(_d)
        if _abs.startswith(os.path.abspath(WORKING_DIR) + os.sep) and os.path.exists(_abs):
            _temp_dirs_to_clean.append(_abs)
    except Exception:
        pass

for _d in _temp_dirs_to_clean:
    if os.path.exists(_d):
        shutil.rmtree(_d, ignore_errors=True)
        print(f"[*] Đã xóa thư mục dữ liệu tạm: {_d}")

cleanup_memory()
print(f"[*] Host RAM sau dọn dẹp -> {format_host_ram()}")